# 11 — Report Figures

Generates every figure for the project report from the saved result files.
All figures are written to `report_figures/` as PNGs (150 dpi) with a
consistent style. Run top to bottom.

In [1]:
import json, os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

os.makedirs('report_figures', exist_ok=True)
mpl.rcParams.update({'figure.dpi':150,'savefig.dpi':150,'font.size':11,
    'axes.titlesize':13,'axes.labelsize':11,'axes.grid':True,'grid.alpha':0.3,
    'axes.axisbelow':True})
COLORS = {'HPA':'#dc2626','DQN':'#f59e0b','PPO':'#2563eb'}

def load(fn, default=None):
    try: return json.load(open(fn))
    except FileNotFoundError:
        print(f"  (missing: {fn})"); return default

def save(fig, name):
    fig.savefig(f'report_figures/{name}.png', bbox_inches='tight')
    plt.close(fig); print(f"  saved report_figures/{name}.png")

print("Setup ready. Figures will go to report_figures/")

Setup ready. Figures will go to report_figures/


In [2]:
# Fig 1 — weekly arrival pattern (Google trace)
tp = load('trace_params.json')['stats']
days = list(range(7))
totals = [sum(tp[str(d)][str(h)]['arrival_rate'] for h in range(24)) for d in days]
fig, ax = plt.subplots(figsize=(8,4))
ax.bar(days, totals, color='#2563eb', alpha=0.85)
ax.set_xlabel('Day of week'); ax.set_ylabel('Total job arrivals')
ax.set_title('Figure 1: Weekly Job Arrival Pattern (Google Cluster Trace 2011)')
ax.set_xticks(days); ax.set_xticklabels([f'Day {d}' for d in days])
save(fig, 'fig01_weekly_arrivals')

  saved report_figures/fig01_weekly_arrivals.png


In [3]:
# Fig 2 — hourly arrival heatmap (day x hour)
mat = np.array([[tp[str(d)][str(h)]['arrival_rate'] for h in range(24)] for d in range(7)])
fig, ax = plt.subplots(figsize=(11,3.5))
im = ax.imshow(mat, aspect='auto', cmap='viridis')
ax.set_xlabel('Hour of day'); ax.set_ylabel('Day')
ax.set_yticks(range(7)); ax.set_yticklabels([f'Day {d}' for d in range(7)])
ax.set_xticks(range(0,24,2))
ax.set_title('Figure 2: Job Arrivals by Day and Hour')
fig.colorbar(im, ax=ax, label='Arrivals/hour')
save(fig, 'fig02_hourly_heatmap')

  saved report_figures/fig02_hourly_heatmap.png


In [4]:
# Fig 3 — PPO training convergence (all 4 configs)
fig, ax = plt.subplots(figsize=(9,5))
cfgnames = ['cost-focused','balanced-cost','balanced-sla','sla-focused']
palette = ['#f59e0b','#10b981','#8b5cf6','#2563eb']
for name, col in zip(cfgnames, palette):
    c = load(f'ppo_{name}_convergence.json', [])
    if c:
        steps = [p['steps'] for p in c]; rew = [p['reward'] for p in c]
        ax.plot(steps, rew, label=name, color=col, linewidth=1.3)
ax.set_xlabel('Training steps'); ax.set_ylabel('Recent episode reward')
ax.set_title('Figure 3: PPO Training Convergence (four reward configurations)')
ax.legend()
save(fig, 'fig03_ppo_convergence')

  saved report_figures/fig03_ppo_convergence.png


In [5]:
# Fig 4 — DQN training convergence
c = load('dqn_convergence.json', [])
fig, ax = plt.subplots(figsize=(9,5))
if c:
    ax.plot([p['steps'] for p in c], [p['reward'] for p in c],
            color='#f59e0b', linewidth=1.4)
ax.set_xlabel('Training steps'); ax.set_ylabel('Recent episode reward')
ax.set_title('Figure 4: DQN Baseline Training Convergence')
save(fig, 'fig04_dqn_convergence')

  saved report_figures/fig04_dqn_convergence.png


In [6]:
# Fig 5 — PPO vs HPA (from significance summary)
summ = load('significance_summary.json')
metrics = ['cost','breaches']
labels = ['Total cost','SLA breaches']
fig, axes = plt.subplots(1, 2, figsize=(10,4.5))
for ax, m, lab in zip(axes, metrics, labels):
    vals = [summ['HPA'][m]['mean'], summ['PPO'][m]['mean']]
    errs = [summ['HPA'][m]['std'], summ['PPO'][m]['std']]
    ax.bar(['HPA','PPO'], vals, yerr=errs, capsize=5,
           color=[COLORS['HPA'],COLORS['PPO']], alpha=0.85)
    ax.set_title(lab); ax.set_ylabel(lab)
fig.suptitle('Figure 5: PPO vs HPA Baseline (mean ± std, n=30 seeds)', y=1.02)
save(fig, 'fig05_ppo_vs_hpa')

  saved report_figures/fig05_ppo_vs_hpa.png


In [7]:
# Fig 6 — three-way HPA/DQN/PPO
fig, axes = plt.subplots(1, 2, figsize=(11,4.5))
for ax, m, lab in zip(axes, ['cost','breaches'], ['Total cost','SLA breaches']):
    agents = ['HPA','DQN','PPO']
    vals = [summ[a][m]['mean'] for a in agents]
    errs = [summ[a][m]['std'] for a in agents]
    ax.bar(agents, vals, yerr=errs, capsize=5,
           color=[COLORS[a] for a in agents], alpha=0.85)
    ax.set_title(lab); ax.set_ylabel(lab)
fig.suptitle('Figure 6: Three-Way Comparison — HPA vs DQN vs PPO (n=30)', y=1.02)
save(fig, 'fig06_three_way')

  saved report_figures/fig06_three_way.png


In [8]:
# Fig 7 — Pareto frontier
pareto = load('pareto_results.json', [])
fig, ax = plt.subplots(figsize=(8,5.5))
if pareto:
    costs = [r['cost'] for r in pareto]; br = [r['breaches'] for r in pareto]
    names = [r['name'] for r in pareto]
    order = np.argsort(costs)
    ax.plot([costs[i] for i in order], [br[i] for i in order],
            'o-', color='#2563eb', label='PPO configs', markersize=8)
    for c,b,n in zip(costs, br, names):
        ax.annotate(n, (c,b), textcoords='offset points', xytext=(8,6), fontsize=9)
# HPA point
hpa = load('significance_summary.json')['HPA']
ax.scatter([hpa['cost']['mean']],[hpa['breaches']['mean']], color=COLORS['HPA'],
           s=120, zorder=5, label='HPA baseline', marker='X')
ax.annotate('HPA', (hpa['cost']['mean'],hpa['breaches']['mean']),
            textcoords='offset points', xytext=(8,6), fontsize=9, color=COLORS['HPA'])
ax.set_yscale('log')
ax.set_xlabel('Total cost per week (lower better)')
ax.set_ylabel('SLA breaches (log scale, lower better)')
ax.set_title('Figure 7: Cost–SLA Pareto Frontier')
ax.legend()
save(fig, 'fig07_pareto_frontier')

  saved report_figures/fig07_pareto_frontier.png


In [18]:
# Fig 8 — significance box plots (30-seed distributions)
res = load('significance_results.json')
fig, axes = plt.subplots(1, 2, figsize=(11,4.5))
for ax, m, lab in zip(axes, ['cost','breaches'], ['Total cost','SLA breaches']):
    data = [[r[m] for r in res[a]] for a in ['HPA','DQN','PPO']]
    bp = ax.boxplot(data, tick_labels=['HPA','DQN','PPO'], patch_artist=True)
    for patch, a in zip(bp['boxes'], ['HPA','DQN','PPO']):
        patch.set_facecolor(COLORS[a]); patch.set_alpha(0.6)
    ax.set_title(lab); ax.set_ylabel(lab)
fig.suptitle('Figure 8: Metric Distributions over 30 Seeds', y=1.02)
save(fig, 'fig08_significance_boxplots')

  saved report_figures/fig08_significance_boxplots.png


In [10]:
# Fig 9 — PPO % improvement over HPA
res = load('significance_results.json')
fig, ax = plt.subplots(figsize=(7,4.5))
imp = {}
for m in ['cost','breaches']:
    p = np.array([r[m] for r in res['PPO']]); h = np.array([r[m] for r in res['HPA']])
    pct = (h-p)/h*100
    imp[m] = (pct.mean(), pct.std())
labels = ['Cost','SLA breaches']
means = [imp['cost'][0], imp['breaches'][0]]
errs = [imp['cost'][1], imp['breaches'][1]]
ax.bar(labels, means, yerr=errs, capsize=6, color='#2563eb', alpha=0.85)
ax.set_ylabel('% reduction vs HPA')
ax.set_title('Figure 9: PPO Improvement over HPA (mean ± std, n=30)')
for i,(mn,er) in enumerate(zip(means,errs)):
    ax.text(i, mn+1, f'{mn:.1f}%', ha='center', fontweight='bold')
save(fig, 'fig09_ppo_improvement')

  saved report_figures/fig09_ppo_improvement.png


In [11]:
# Fig 10 — inference latency and resource overhead
sm = load('system_metrics.json')
fig, axes = plt.subplots(1, 2, figsize=(11,4.5))
agents = ['HPA','DQN','PPO']
lat = [sm['latency_ms'][a] for a in agents]
axes[0].bar(agents, lat, color=[COLORS[a] for a in agents], alpha=0.85)
axes[0].set_ylabel('Inference latency (ms)')
axes[0].set_title('Inference Latency per Decision')
par = [sm['params'][a]/1000 for a in agents]
axes[1].bar(agents, par, color=[COLORS[a] for a in agents], alpha=0.85)
axes[1].set_ylabel('Parameters (thousands)')
axes[1].set_title('Model Size (parameters)')
fig.suptitle('Figure 10: Computational Overhead', y=1.02)
save(fig, 'fig10_latency_overhead')

  saved report_figures/fig10_latency_overhead.png


In [12]:
# Fig 11 — Alibaba generalisation: VM + breaches over time
ts = load('alibaba_timeseries.json')
if ts:
    vms = ts['vms']; br = ts['breaches']
    hours = np.arange(len(vms))*15/60
    fig,(a1,a2)=plt.subplots(2,1,figsize=(11,6),sharex=True,gridspec_kw={'height_ratios':[2,1]})
    a1.plot(hours, vms, color='#2563eb', linewidth=1.2)
    a1.fill_between(hours, vms, alpha=0.15, color='#2563eb')
    a1.axhline(2, color='gray', ls='--', lw=0.7); a1.axhline(20, color='red', ls='--', lw=0.7)
    a1.set_ylabel('Active VMs'); a1.set_ylim(0,21)
    a1.set_title('Figure 11: PPO on Unseen Alibaba Workload (Google-trained, no retraining)')
    a2.fill_between(hours, br, color='#dc2626', alpha=0.5)
    a2.set_ylabel('SLA breaches'); a2.set_xlabel('Time (hours over one week)')
    for d in range(1,7):
        a1.axvline(d*24, color='gray', lw=0.4, alpha=0.4)
        a2.axvline(d*24, color='gray', lw=0.4, alpha=0.4)
    save(fig, 'fig11_alibaba_vm_timeseries')

  saved report_figures/fig11_alibaba_vm_timeseries.png


In [13]:
# Fig 12 — Alibaba PPO vs HPA
ac = load('alibaba_comparison.json')
if ac:
    fig, axes = plt.subplots(1,3, figsize=(12,4))
    for ax,m,lab in zip(axes,['cost','breaches','vms'],['Total cost','SLA breaches','Avg VMs']):
        ax.bar(['HPA','PPO'], [ac['hpa'][m], ac['ppo'][m]],
               color=[COLORS['HPA'],COLORS['PPO']], alpha=0.85)
        ax.set_title(lab)
    fig.suptitle('Figure 12: Cross-Provider Generalisation on Alibaba 2018', y=1.03)
    save(fig, 'fig12_alibaba_comparison')

  saved report_figures/fig12_alibaba_comparison.png


In [14]:
# Fig 13 — surge behaviour (queue + breaches)
st = load('surge_timeseries.json')
if st:
    q = st['queue']; br = st['breaches']; sm_ = st['surge']
    steps = np.arange(len(q))
    # zoom to the surge window
    surge_steps = [i for i,s in enumerate(sm_) if s>1.0]
    if surge_steps:
        lo=max(0,surge_steps[0]-10); hi=min(len(q),surge_steps[-1]+20)
    else:
        lo,hi=0,len(q)
    fig,(a1,a2)=plt.subplots(2,1,figsize=(10,6),sharex=True)
    a1.plot(steps[lo:hi], q[lo:hi], color='#2563eb', linewidth=1.4)
    a1.axvspan(surge_steps[0], surge_steps[-1], color='orange', alpha=0.2, label='surge window')
    a1.set_ylabel('Queue length'); a1.set_title('Figure 13: Cluster Response to a Traffic Surge')
    a1.legend()
    a2.fill_between(steps[lo:hi], br[lo:hi], color='#dc2626', alpha=0.5)
    a2.set_ylabel('SLA breaches'); a2.set_xlabel('Simulation step')
    save(fig, 'fig13_surge_behaviour')

  saved report_figures/fig13_surge_behaviour.png


In [15]:
# Fig 14 — hint bracketing (±5 vs ±1)
d5 = load('hint_result_dense.json'); d1 = load('hint_result_constrained.json')
if d5 and d1:
    fig, ax = plt.subplots(figsize=(7,4.5))
    regimes = ['±5 VM/step\n(fast reactive)', '±1 VM/step\n(constrained)']
    reductions = [d5['reduction_pct'], d1['reduction_pct']]
    bars = ax.bar(regimes, reductions, color=['#2563eb','#f59e0b'], alpha=0.85)
    ax.set_ylabel('Hint benefit (% breach reduction)')
    ax.set_title('Figure 14: Operator-Hint Benefit by Scaling Regime (bracketing)')
    for b,r in zip(bars, reductions):
        ax.text(b.get_x()+b.get_width()/2, r+0.1, f'{r:.1f}%', ha='center', fontweight='bold')
    save(fig, 'fig14_hint_bracketing')

  saved report_figures/fig14_hint_bracketing.png


In [19]:
import os
figs = sorted(os.listdir('report_figures'))
print(f"\nGenerated {len(figs)} figures in report_figures/:")
for f in figs: print("  ", f)


Generated 14 figures in report_figures/:
   fig01_weekly_arrivals.png
   fig02_hourly_heatmap.png
   fig03_ppo_convergence.png
   fig04_dqn_convergence.png
   fig05_ppo_vs_hpa.png
   fig06_three_way.png
   fig07_pareto_frontier.png
   fig08_significance_boxplots.png
   fig09_ppo_improvement.png
   fig10_latency_overhead.png
   fig11_alibaba_vm_timeseries.png
   fig12_alibaba_comparison.png
   fig13_surge_behaviour.png
   fig14_hint_bracketing.png
